# Base 0 (Zero-Shot) Inference Orchestrator

Este notebook realiza el clonado sparse del repositorio, instalación de dependencias en modo editable y orquesta la inferencia y recolección de métricas para la **Base 0**.

**Estructura esperada del dataset en Kaggle** (`mtc-challenge`):
```
/kaggle/input/mtc-challenge/
├── train-001/          ← zip auto-extraído
│   └── train/          ← carpeta dentro del zip
│       ├── clip_id_1/
│       │   ├── 0001.jpg
│       │   └── ...
│       └── clip_id_2/
├── split_metadata.csv
└── train.csv
```

In [ ]:
import os
from pathlib import Path

REPO_NAME = 'ia_article'
REPO_URL = 'https://github.com/unsa-semester-2026-A/ia_article.git'
BRANCH_NAME = 'feat/19-base0-evaluation'

# --- Detectar entorno: Kaggle vs Colab ---
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
%cd {BASE_DIR}

REPO_PATH = Path(BASE_DIR) / REPO_NAME

# 1. Clone repository sparsely from feature branch
if not REPO_PATH.exists():
    print(f"Clonando {REPO_NAME} (rama {BRANCH_NAME})...")
    !git clone -q --depth 1 --branch {BRANCH_NAME} --filter=blob:none --sparse {REPO_URL}
    %cd {REPO_NAME}
    !git sparse-checkout set experiments
    %cd experiments
else:
    print(f"El repositorio ya existe. Actualizando {REPO_NAME}...")
    %cd {REPO_NAME}
    !git checkout {BRANCH_NAME}
    !git pull -q origin {BRANCH_NAME}
    %cd experiments

# 2. Verify working directory
current_dir = Path(os.getcwd())
if current_dir.name != 'experiments':
    raise RuntimeError(f"Fallo al navegar al directorio. Ruta actual: {current_dir}")

# 3. Install dependencies in editable mode
print("Instalando el paquete en modo editable con dependencias [cloud]...")
%pip install -q -e .[cloud]

## 2. Configuración y Ejecución del Pipeline Base 0

La configuración detecta automáticamente el entorno y descarga automáticamente `token.json` desde Google Drive.

In [ ]:
import os
import sys
import time
import urllib.request
from pathlib import Path
from src.inference.runners.run_base_0 import Base0Runner

# --- Detección dinámica del entorno ---
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = os.path.exists('/content/drive')

if IS_KAGGLE:
    DATASET_DIR = Path('/kaggle/input/mtc-challenge')
    OUTPUT_DIR = Path('/kaggle/working/output_base0')
    TOKEN_PATH = Path('/kaggle/working/token.json')
else:
    DATASET_DIR = Path('/content/drive/MyDrive/ia_article')
    OUTPUT_DIR = Path('/content/output_base0')
    TOKEN_PATH = Path('/content/token.json')

# --- Descarga automática del token.json desde Google Drive --- 
TOKEN_FILE_ID = '1Fjg-AIrIQ77g1JRtapE6XDb_A6CP_4q3'
if not TOKEN_PATH.exists():
    print(f"Descargando token.json automáticamente desde Google Drive...")
    token_url = f'https://drive.google.com/uc?export=download&id={TOKEN_FILE_ID}'
    try:
        urllib.request.urlretrieve(token_url, str(TOKEN_PATH))
        print(f"✅ token.json guardado en {TOKEN_PATH}")
    except Exception as e:
        print(f"⚠️  No se pudo descargar token.json automáticamente: {e}")

# --- Rutas según estructura del dataset de Kaggle ---
config = {
    "device": 0,
    "conf": 0.001,
    "iou": 0.45,
    "imgsz": 640,
    "batch_size": 16,
    "metadata_path": str(DATASET_DIR / "split_metadata.csv"),
    "images_dir": str(DATASET_DIR / "train-001" / "train"),
    "output_dir": str(OUTPUT_DIR),
    "hardware_name": "Tesla_T4_Kaggle" if IS_KAGGLE else "Colab_GPU",
    "experiment_condition": "Base_0_Zero_Shot",
    "token_path": str(TOKEN_PATH) if TOKEN_PATH.exists() else None,
    "drive_folder_id": "1wXieZvOZDE5KzZiYyESbf8xU-C2AGPWJ",
}

# --- Verificar que las rutas existan ---
metadata = Path(config["metadata_path"])
images = Path(config["images_dir"])
print(f"Entorno: {'Kaggle' if IS_KAGGLE else 'Colab'}")
print(f"Metadata: {metadata} -> {'✅ existe' if metadata.exists() else '❌ NO EXISTE'}")
print(f"Imágenes: {images} -> {'✅ existe' if images.exists() else '❌ NO EXISTE'}")
print(f"Token Drive: {config['token_path'] or 'No disponible'}")
print(f"Drive folder ID: {config['drive_folder_id']}")

if not metadata.exists() or not images.exists():
    raise FileNotFoundError("Rutas del dataset no encontradas. Verifica la estructura del dataset.")

In [ ]:
# --- Ejecutar Inferencia ---
print("Inicializando ejecutor Base 0...")
runner = Base0Runner(config=config, model_path="yolo26m-obb.pt")

print("\nEjecutando inferencia (máx. 5 clips de prueba)...")
results = runner.execute()

print("\n" + "="*60)
print("RESULTADO FINAL")
print("="*60)
print(f"Estado: {results['status']}")
print(f"Archivos generados: {len(results['files'])}")
for f in results['files']:
    drive_status = f'Drive ID: {f["drive_id"]}' if f['drive_id'] else 'Solo local'
    print(f"  📄 {Path(f['local']).name} -> {drive_status}")
print(f"\nMétricas:")
for k, v in results['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# --- Apagar sesión para no consumir créditos de cómputo ---
print("\nInferencia completada. Apagando sesión en 5 segundos para salvar créditos...")
sys.stdout.flush()
sys.stderr.flush()
time.sleep(5)

if IS_KAGGLE:
    # En Kaggle no hay API directa de desconexión, pero podemos detener el kernel
    print("Sesión de Kaggle finalizada. Los archivos están en /kaggle/working/output_base0/")
    os._exit(0)
elif IS_COLAB:
    from google.colab import runtime
    print("Desconectando sesión de Google Colab...")
    runtime.unassign()